# Eurostat Energy Balance Evaluation 
Processing of the Eurostat Energy Balance for a specific region/country for the IAMC-variables defined in `/definitions`.

### Data References
- API-url for EB datasource: [https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/nrg_bal_c?format=TSV&compressed=true](https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/nrg_bal_c?format=TSV&compressed=true)
- Definitions of IAMC-Variables: [IIASA/energy-scenarios-at-workflow](https://github.com/iiasa/energy-scenarios-at-workflow/tree/main)

In [19]:
import pandas as pd 
import pyam 
import nomenclature

In [20]:
import os

# Use a path relative to this notebook's location so it works on any machine.
# Adjust the path if the notebook is moved relative to the repository root.
definitions_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "definitions")
dsd = nomenclature.DataStructureDefinition(definitions_path)

In [21]:
df_n = dsd.variable.to_pandas()

### Mapping Dictionaries

Two dictionaries link IAMC variable names/dimensions to Eurostat Energy Balance codes:

#### `siec_dict` — Carrier / Source → Eurostat SIEC codes
Maps each IAMC *carrier* or *electricity-generation-source* name (as used in placeholder tags
like `{Final Energy Carrier}` and `{Electricity Generation by Source}`) to the
corresponding Eurostat `siec` (Standard International Energy Product Classification) codes.

Example:
```
"Natural Gas" → ["G3000"]
"Hydro"        → ["RA100"]
```

#### `nrg_dict` — IAMC variable template → Eurostat NRG_BAL codes
Maps IAMC variable-name *templates* (optionally containing `{…}` placeholders) to the
Eurostat `nrg_bal` (energy-balance item) codes that must be summed.
A leading `-` on a code means the values are subtracted (e.g. exports in net-import
calculations).

Example:
```
"Final Energy" → ["FC_E", "NRG_E"]
"Net Imports|{Final Energy Carrier}" → ["IMP", "-EXP"]
```


In [33]:
siec_dict = {
    "Coal": ["C0000X0350-0370", "P1000"],
    "Oil": ["O4000XBIO"],
    "Natural Gas": ["G3000"],
    "Biomass": ["R5110-5150_W6000RI", "R5160", "R5300"],
    "Biogas": ["R5300"],
    "E-Methane": [],
    "Hydro": ["RA100"],
    "Solar": ["RA420"],
    "Wind": ["RA300"],
    "Geothermal": ["RA200"],
    "Waste": ["W6100_6220"],
    "Other Renewables": ["RA500"],
    "Electricity": ["E7000"],
    "Biofuels": ["R5210P", "R5220P", "R5230P", "R5290", "R5160"],
    "E-Fuels": [],
    "Biomethane": ["R5300"],
    "Hydrogen": [],
    "Solar Thermal": ["RA410"],
    "District Heat": ["H8000"],
    "Ambient Heat": ["RA600"],
    "Other": [],
    "Renewables": [
        "RA100", "RA500", "RA300", "RA420", "RA410", "RA200",
        "R5110-5150_W6000RI", "R5160", "R5300", "W6210",
        "R5210P", "R5210B", "R5220P", "R5220B", "R5230P", "R5230B", "R5290",
    ],
    "Renewables incl. Ambient Heat": ["RA000"],
}

nrg_dict = {
    "Net Imports|{Final Energy Carrier}": ["IMP", "-EXP"],
    "Final Energy": ["FC_E", "NRG_E"],
    "Final Energy [by Carrier]|{Final Energy Carrier}": ["FC_E", "NRG_E"],
    "Final Energy [by Sector]|{Sector}": [],
    "Final Energy [by Sector]|{Sector}|{Final Energy Carrier}": [],
    "Secondary Energy|Electricity": ["GEP"],
    "Secondary Energy|Electricity|{Electricity Generation by Source}": [
        "TI_EHG_MAPE_E", "TI_EHG_MAPCHP_E", "TI_EHG_APE_E", "TI_EHG_APCHP_E",
    ],
}

no direct mapping of variables possible, but multiindex-structure to read out variables of EB to be mapped on the IAMC-Variables.

In [27]:
from energy_balance_evaluation.energy_balance_eval import fetch_and_load_tsv_data

# Load and filter the Eurostat energy balance data for Austria (AT).
# The file "resources/pypsa-eur-download.tsv" is the local Eurostat nrg_bal_c TSV.
# If absent, fetch_and_load_tsv_data will attempt to download it from the Eurostat API.
eb_df = fetch_and_load_tsv_data("resources/pypsa-eur-download.tsv")
eb_df = eb_df[eb_df.geo == "AT"]


In [30]:
eb_df

,freq,nrg_bal,siec,unit,geo,1990,1991,1992,1993,1994,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,A,AFC,BIOE,GWH,AL,4221.944,4221.944,4221.944,4029.722,3846.389,...,2728.383,3049.733,2804.720,3129.617,3211.056,3013.181,2963.553,3180.853,3291.735,3098.084
1,A,AFC,BIOE,GWH,AT,25744.633,27905.111,27110.141,27367.890,25186.310,...,57495.297,57863.539,58217.651,55736.679,55563.334,54830.084,60953.408,59869.408,62017.419,62384.826
2,A,AFC,BIOE,GWH,BA,NaN,NaN,NaN,NaN,NaN,...,5937.175,5936.294,4843.579,13379.486,13990.576,14750.870,14124.598,15190.525,14556.421,14291.273
3,A,AFC,BIOE,GWH,BE,4034.103,4108.185,3958.904,3139.375,3076.269,...,25304.247,27008.692,27175.362,27777.464,25898.526,27773.762,30739.630,30618.106,30145.021,29175.304
4,A,AFC,BIOE,GWH,BG,2005.833,1333.333,1818.057,1656.454,1865.723,...,13750.594,14423.227,14656.419,16327.377,16722.072,18123.927,17830.582,16818.733,15058.769,14226.252
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1216492,A,TO_RPI_RO,W6220,TJ,SK,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1216493,A,TO_RPI_RO,W6220,TJ,TR,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1216494,A,TO_RPI_RO,W6220,TJ,UA,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1216495,A,TO_RPI_RO,W6220,TJ,UK,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
df_n

,variable,description,unit,EB,note
0,Final Energy,Final energy consumption by all end-use sector...,TJ,"FC_E,NRG_E",Final energy consumption explicitly excludes e...
1,Secondary Energy|Electricity,Total net generation of electricity (including...,TWh,GEP,NaN
2,Final Energy [by Carrier]|Coal,Energy consumption of coal excluding transmiss...,TJ,"FC_E,NRG_E",Final energy consumption explicitly excludes e...
3,Final Energy [by Carrier]|Oil,Energy consumption of oil excluding transmissi...,TJ,"FC_E,NRG_E",Final energy consumption explicitly excludes e...
4,Final Energy [by Carrier]|Natural Gas,Energy consumption of fossil methane excluding...,TJ,"FC_E,NRG_E",Final energy consumption explicitly excludes e...
...,...,...,...,...,...
121,Net Imports|District Heat,Net imports of heat delivered from district he...,"[""TJ"", ""TWh""]","IMP,-EXP",EB states imports and exports as positive values.
122,Net Imports|Ambient Heat,"Net imports of ambient heat using heat pumps, ...","[""TJ"", ""TWh""]","IMP,-EXP",EB states imports and exports as positive values.
123,Net Imports|Other,Net imports of other sources or carriers,"[""TJ"", ""TWh""]","IMP,-EXP",EB states imports and exports as positive values.
124,Net Imports|Renewables,Net imports of renewables including biomass an...,"[""TJ"", ""TWh""]","IMP,-EXP",EB states imports and exports as positive values.


### IAMC variable calculation from Eurostat energy balance

This cell computes IAMC-style variable values by combining `eb_df` with the variable definitions in `df_n` using the mapping dictionaries `siec_dict` and `nrg_dict`.

- **Input data**:
  - `eb_df` (Eurostat energy-balance data, already filtered to a region, e.g. `AT`)
  - `df_n` (variable definition table from the nomenclature DSD)
  - `siec_dict` (maps IAMC carrier/source names to Eurostat `siec` codes)
  - `nrg_dict` (maps IAMC variable templates to Eurostat `nrg_bal` codes)

- **How matching works**:
  - Each variable name from `df_n` is matched against template keys in `nrg_dict` (including placeholders like `{Final Energy Carrier}` or `{Electricity Generation by Source}`).
  - Placeholder values are resolved via `siec_dict` to obtain one or more `siec` codes.
  - The selected `nrg_bal` rows are aggregated over all detected year columns.
  - Signed entries in `nrg_dict` (e.g. `-EXP`) are subtracted.

- **Missing/incomplete mappings**:
  - If no template match exists in `nrg_dict`, the result is `np.nan`.
  - If a matched `nrg_dict` entry is empty, the result is `np.nan`.
  - If a required `siec_dict` entry is empty/missing, the result is `np.nan`.
  - If no matching rows are found in `eb_df`, the result is `np.nan`.

- **Outputs**:
  - `calculated_values_df`: computed values per IAMC variable and year.
  - `df_n_with_values`: original `df_n` plus the computed year columns.

In [34]:
import re
import numpy as np
import pandas as pd

year_cols = [col for col in eb_df.columns if re.fullmatch(r"\d{4}", str(col))]
if not year_cols:
    year_cols = [
        col
        for col in eb_df.columns
        if pd.api.types.is_numeric_dtype(eb_df[col]) and col not in {"nrg_bal", "siec"}
    ]

if not year_cols:
    raise ValueError("No year columns found in eb_df.")

def _get_variable_list(df_variables: pd.DataFrame) -> list[str]:
    if "name" in df_variables.columns:
        return df_variables["name"].astype(str).tolist()
    if "variable" in df_variables.columns:
        return df_variables["variable"].astype(str).tolist()
    return df_variables.index.astype(str).tolist()

def _match_nrg_template(variable_name: str, mapping: dict[str, list[str]]) -> tuple[str | None, dict[str, str]]:
    for template in mapping:
        pattern = "^" + re.escape(template) + "$"
        placeholders = re.findall(r"\{([^}]+)\}", template)
        for placeholder in placeholders:
            pattern = pattern.replace(r"\{" + re.escape(placeholder) + r"\}", r"(.+?)", 1)

        match = re.match(pattern, variable_name)
        if match:
            values = {placeholder: match.group(i + 1) for i, placeholder in enumerate(placeholders)}
            return template, values

    return None, {}

def _get_siec_codes(placeholder_values: dict[str, str]) -> list[str] | None:
    siec_codes: list[str] = []

    for placeholder, value in placeholder_values.items():
        if placeholder in {"Final Energy Carrier", "Electricity Generation by Source"}:
            codes = siec_dict.get(value, [])
            if not codes:
                return None
            siec_codes.extend(codes)
        elif placeholder == "Sector":
            return None

    return sorted(set(siec_codes)) if siec_codes else []

def _calculate_single_variable(variable_name: str) -> pd.Series:
    template, placeholder_values = _match_nrg_template(variable_name, nrg_dict)
    if template is None:
        return pd.Series(np.nan, index=year_cols, dtype=float)

    nrg_codes = nrg_dict.get(template, [])
    if not nrg_codes:
        return pd.Series(np.nan, index=year_cols, dtype=float)

    siec_codes = _get_siec_codes(placeholder_values)
    if siec_codes is None:
        return pd.Series(np.nan, index=year_cols, dtype=float)

    result = pd.Series(0.0, index=year_cols, dtype=float)
    has_data = False

    for code in nrg_codes:
        sign = -1.0 if str(code).startswith("-") else 1.0
        clean_code = str(code).lstrip("-")

        subset = eb_df.loc[eb_df["nrg_bal"] == clean_code].copy()
        if siec_codes:
            subset = subset.loc[subset["siec"].isin(siec_codes)]

        if subset.empty:
            continue

        values = subset[year_cols].apply(pd.to_numeric, errors="coerce").sum(axis=0, min_count=1)
        result = result.add(sign * values, fill_value=np.nan)
        has_data = True

    if not has_data:
        return pd.Series(np.nan, index=year_cols, dtype=float)

    return result

variable_list = _get_variable_list(df_n)

calculated_records: list[dict[str, object]] = []
for variable in variable_list:
    value_series = _calculate_single_variable(variable)
    record = {"variable": variable, **value_series.to_dict()}
    calculated_records.append(record)

calculated_values_df = pd.DataFrame(calculated_records).set_index("variable")

if "name" in df_n.columns:
    df_n_with_values = df_n.merge(calculated_values_df, left_on="name", right_index=True, how="left")
elif "variable" in df_n.columns:
    df_n_with_values = df_n.merge(calculated_values_df, on="variable", how="left")
else:
    df_n_with_values = df_n.join(calculated_values_df, how="left")

df_n_with_values.head()

,variable,description,unit,EB,note,1990,1991,1992,1993,1994,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Final Energy,Final energy consumption by all end-use sector...,TJ,"FC_E,NRG_E",Final energy consumption explicitly excludes e...,4.419315e+08,4.417019e+08,4.276733e+08,4.247500e+08,4.179686e+08,...,4.197387e+08,4.279501e+08,4.343776e+08,4.343293e+08,4.310277e+08,3.831880e+08,3.972268e+08,3.846585e+08,3.723016e+08,3.508572e+08
1,Secondary Energy|Electricity,Total net generation of electricity (including...,TWh,GEP,NaN,7.998978e+07,8.058512e+07,7.955800e+07,7.851421e+07,7.932387e+07,...,9.776066e+07,9.839688e+07,9.917349e+07,9.858770e+07,9.571726e+07,8.780304e+07,9.026105e+07,8.874047e+07,8.575327e+07,8.143777e+07
2,Final Energy [by Carrier]|Coal,Energy consumption of coal excluding transmiss...,TJ,"FC_E,NRG_E",Final energy consumption explicitly excludes e...,1.181613e+07,1.033920e+07,8.960666e+06,8.423486e+06,7.444407e+06,...,3.762030e+06,3.802285e+06,3.762769e+06,3.564321e+06,3.249631e+06,3.138809e+06,2.920276e+06,2.561617e+06,2.263197e+06,1.494353e+06
3,Final Energy [by Carrier]|Oil,Energy consumption of oil excluding transmissi...,TJ,"FC_E,NRG_E",Final energy consumption explicitly excludes e...,5.127986e+07,5.215348e+07,5.127205e+07,5.116059e+07,5.090385e+07,...,4.581758e+07,4.656384e+07,4.705498e+07,4.668221e+07,4.674534e+07,3.959449e+07,4.124909e+07,4.234291e+07,4.163880e+07,3.946792e+07
4,Final Energy [by Carrier]|Natural Gas,Energy consumption of fossil methane excluding...,TJ,"FC_E,NRG_E",Final energy consumption explicitly excludes e...,2.386414e+07,2.502034e+07,2.387357e+07,2.425687e+07,2.431389e+07,...,2.696228e+07,2.783387e+07,2.827528e+07,2.838804e+07,2.810141e+07,2.509630e+07,2.672881e+07,2.289304e+07,2.153889e+07,2.020705e+07


In [ ]:
# Display computed results
# `calculated_values_df` contains computed IAMC variable values indexed by variable name
# with one column per year available in the energy balance.
calculated_values_df

### Next Steps

- `calculated_values_df` can be passed to `pyam.IamDataFrame` together with model/scenario
  metadata to produce a valid IAMC object for further validation.
- To generate the reference YAML codelists used by `nomenclature`-based validation,
  use `VariablesSet.write_codelist()` from `energy_balance_evaluation.energy_balance_eval`.
- Missing values (`NaN`) indicate that either no mapping was defined, the carrier/source
  name has no SIEC codes in `siec_dict`, or no matching rows were found in the energy
  balance for the selected country.
